# Conditionals and Iteration

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
# warnings.simplefilter(action='ignore', category=np.VisibleDeprecationWarning)

In [ ]:
# polars
import polars as pl
from IPython.display import display
import numpy as np

def _append(base, extra):
    """Append a value or a Series onto a Series."""
    extra_s = extra if isinstance(extra, pl.Series) else pl.Series([extra])
    if base.len() == 0:
        return extra_s
    if base.dtype != extra_s.dtype:
        try:
            extra_s = extra_s.cast(base.dtype, strict=False)
        except Exception:
            base = base.cast(extra_s.dtype, strict=False)
    return pl.concat([base, extra_s])


## Appending Arrays

In [ ]:
# datascience
first = np.arange(4)
second = np.arange(10, 17)

In [ ]:
# polars
first_pl = pl.int_range(4, eager=True)
second_pl = pl.int_range(10, 17, eager=True)

In [ ]:
# datascience
first

In [ ]:
# polars
first_pl

In [ ]:
# datascience
np.append(first, 6)

In [ ]:
# polars
_append(first_pl, 6)

In [ ]:
# datascience
first

In [ ]:
# polars
first_pl

In [ ]:
# datascience
# how do we change `first`?
first = np.append(first, 6)

In [ ]:
# polars
# how do we change `first`?
first_pl = _append(first_pl, 6)

In [ ]:
# datascience
np.append(first, second)

In [ ]:
# polars
_append(first_pl, second_pl)

In [ ]:
# datascience
first

In [ ]:
# polars
first_pl

In [ ]:
# datascience
second

In [ ]:
# polars
second_pl

## Comparison ##

In [ ]:
3 > 1

In [ ]:
type(3 > 1)

In [ ]:
True

In [ ]:
true

(We see that capitalization matters.)

In [ ]:
3 = 3

In [ ]:
3 == 3.0

In [ ]:
10 != 2

In [ ]:
x = 14
y = 3

In [ ]:
x > 15

In [ ]:
12 < x

In [ ]:
x < 20

In [ ]:
12 < x < 20

In [ ]:
10 < x-y < 13

In [ ]:
x > 13 and y < 3.14159

(The comparison `12 < x < 20` is equivalent to `12 < x and x < 20`.)

## Comparisons with arrays

In [ ]:
# datascience
pets = make_array('cat', 'cat', 'dog', 'cat', 'dog', 'rabbit')

In [ ]:
# polars
pets_pl = pl.Series("values", ['cat', 'cat', 'dog', 'cat', 'dog', 'rabbit'])

In [ ]:
# datascience
pets == 'cat'

In [ ]:
# polars
pets_pl == 'cat'

In [ ]:
1 + 1 + 0 + 1 + 0 + 0

In [ ]:
# datascience
sum(make_array(True, True, False, True, False, False))

In [ ]:
# polars
sum(pl.Series("values", [True, True, False, True, False, False]))

In [ ]:
# datascience
sum(pets == 'cat')

In [ ]:
# polars
sum(pets_pl == 'cat')

In [ ]:
# datascience
np.count_nonzero(pets == 'cat')

In [ ]:
# polars
(lambda _s: _s.sum() if _s.dtype == pl.Boolean else (_s != 0).sum())(pets_pl == 'cat')

In [ ]:
# datascience
xs = np.arange(20, 31)

In [ ]:
# polars
xs_pl = pl.int_range(20, 31, eager=True)

In [ ]:
# datascience
xs

In [ ]:
# polars
xs_pl

In [ ]:
# datascience
xs > 28

In [ ]:
# polars
xs_pl > 28

In [ ]:
# datascience
sum(xs > 28)

In [ ]:
# polars
sum(xs_pl > 28)

## Conditional Statements

In [ ]:
age = 20

In [ ]:
if age >= 18:
    print('You can legally vote.')
else:
    if age >= 16:
        print("You can't vote just yet, but you can pre-register!")
    else:
        print("You can't vote just yet.")

In [ ]:
if age >= 18:
    print('You can legally vote.')
if age >= 16:
    print("You can't vote just yet, but you can pre-register!")
else:
    print("You can't vote just yet.")

In [ ]:
def vote(age):
    if age >= 18:
        return 'You can legally vote.'
    elif age >= 16:
        return "You can't vote just yet, but you can pre-register!"
    else:
        return "You can't vote just yet."

In [ ]:
vote(3)

In [ ]:
vote(17)

In [ ]:
vote(25)

In [ ]:
# datascience
def rights(age):
	my_rights = make_array()
	if age >= 16:
		my_rights = np.append(my_rights, 'register to vote')
	if age >= 18:
		my_rights = np.append(my_rights, 'vote')
	return my_rights

In [ ]:
# polars
def rights_pl(age):
	my_rights = pl.Series("values", [], dtype=pl.String)
	if age >= 16:
		my_rights = _append(my_rights, 'register to vote')
	if age >= 18:
		my_rights = _append(my_rights, 'vote')
	return my_rights

In [ ]:
# datascience
rights(3)

In [ ]:
# polars
rights_pl(3)

In [ ]:
# datascience
rights(17)

In [ ]:
# polars
rights_pl(17)

In [ ]:
# datascience
rights(25)

In [ ]:
# polars
rights_pl(25)

In [ ]:
# datascience
trip = Table().read_table('trip.csv').sort('Zip Code')
trip.show(5)

In [ ]:
# polars
trip_pl = pl.read_csv('trip.csv', schema_overrides={"Zip Code": pl.String}).sort('Zip Code')
trip_pl.head(5)

In [ ]:
# datascience
def trip_kind(start, end):
    if start == end:
        return 'round trip'
    else:
        return 'one way'

# trip.show(5)

kinds = trip.apply(trip_kind, 'Start Station', 'End Station')
with_kinds = trip.with_column('Trip Kind', kinds)
with_kinds.show(5)

In [ ]:
# polars
def trip_kind_pl(start, end):
    if start == end:
        return 'round trip'
    else:
        return 'one way'

# trip.show(5)

kinds_pl = trip_pl.select(pl.struct(['Start Station', 'End Station']).map_elements(lambda s: trip_kind_pl(s['Start Station'], s['End Station']), return_dtype=pl.String)).to_series()
with_kinds_pl = trip_pl.with_columns(pl.Series('Trip Kind', kinds_pl))
with_kinds_pl.head(5)

In [ ]:
# datascience
# recall pivot! what will we see?
with_kinds.where('Duration', are.below(600)).pivot('Trip Kind', 'Start Station')

In [ ]:
# polars
# recall pivot! what will we see?
(lambda _p: _p.select(['Start Station'] + sorted([c for c in _p.columns if c != 'Start Station'], key=str)))(with_kinds_pl.filter(pl.col('Duration') < 600).pivot('Trip Kind', index='Start Station', values='Trip Kind', aggregate_function='len').sort('Start Station'))

## Simulation

Let's play a game: we each roll a die. 

If my number is bigger: you pay me a dollar.

If they're the same: we do nothing.

If your number is bigger: I pay you a dollar.

Steps:
1. Find a way to simulate two dice rolls.
2. Compute how much money we win/lose based on the result.
3. Do steps 1 and 2 10,000 times.

### Conditional Statements

In [ ]:
# Work in progress
def one_round(my_roll, your_roll):
    if my_roll > your_roll:
        return 1

In [ ]:
one_round(4, 3)

In [ ]:
one_round(2, 6)

In [ ]:
def one_round(my_roll, your_roll):
    if my_roll > your_roll:
        return 1
    elif your_roll > my_roll:
        return -1
    elif your_roll == my_roll:
        return 0

In [ ]:
one_round(1, 1)

In [ ]:
one_round(6, 5)

In [ ]:
one_round(-7, -1)

### Random Selection

In [ ]:
# datascience
mornings = make_array('wake up', 'sleep in')

In [ ]:
# polars
mornings_pl = pl.Series("values", ['wake up', 'sleep in'])

In [ ]:
# datascience
np.random.choice(mornings)

In [ ]:
# polars
(mornings_pl).sample(1).item()

In [ ]:
# datascience
np.random.choice(mornings)

In [ ]:
# polars
(mornings_pl).sample(1).item()

In [ ]:
# datascience
np.random.choice(mornings)

In [ ]:
# polars
(mornings_pl).sample(1).item()

We can also pass an argument that specifies how many times to make a random choice:

In [ ]:
# datascience
np.random.choice(mornings, 7)

In [ ]:
# polars
(mornings_pl).sample(7, with_replacement=True)

In [ ]:
# datascience
sum(np.random.choice(mornings, 7) == 'wake up')

In [ ]:
# polars
sum((mornings_pl).sample(7, with_replacement=True) == 'wake up')

In [ ]:
# datascience
sum(np.random.choice(mornings, 7) == 'sleep in')

In [ ]:
# polars
sum((mornings_pl).sample(7, with_replacement=True) == 'sleep in')

^ Why don't these (always) sum to 7?

In [ ]:
# datascience
morning_week = np.random.choice(mornings, 7)
morning_week

In [ ]:
# polars
morning_week_pl = (mornings_pl).sample(7, with_replacement=True)
morning_week_pl

In [ ]:
# datascience
sum(morning_week == 'wake up')

In [ ]:
# polars
sum(morning_week_pl == 'wake up')

In [ ]:
# datascience
sum(morning_week == 'sleep in')

In [ ]:
# polars
sum(morning_week_pl == 'sleep in')

### Simulating the roll of a die

In [ ]:
# datascience
die_faces = np.arange(1, 7)

In [ ]:
# polars
die_faces_pl = pl.int_range(1, 7, eager=True)

In [ ]:
# datascience
np.random.choice(die_faces)

In [ ]:
# polars
(die_faces_pl).sample(1).item()

In [ ]:
# datascience
def simulate_one_round():
    my_roll = np.random.choice(die_faces)
    your_roll = np.random.choice(die_faces)
    return one_round(my_roll, your_roll)

In [ ]:
# polars
def simulate_one_round_pl():
    my_roll = (die_faces_pl).sample(1).item()
    your_roll = (die_faces_pl).sample(1).item()
    return one_round(my_roll, your_roll)

In [ ]:
# datascience
simulate_one_round()

In [ ]:
# polars
simulate_one_round_pl()

### Repeated Betting ###

In [ ]:
# datascience
results = make_array()

In [ ]:
# polars
results_pl = pl.Series("values", [], dtype=pl.Float64)

In [ ]:
# datascience
results = np.append(results, simulate_one_round())
results

In [ ]:
# polars
results_pl = _append(results_pl, simulate_one_round_pl())
results_pl

In [ ]:
# datascience
results = np.append(results, simulate_one_round())
results

In [ ]:
# polars
results_pl = _append(results_pl, simulate_one_round_pl())
results_pl

## `For` Statements

In [ ]:
# datascience
for pet in make_array('cat', 'dog', 'rabbit'):
    print('I love my ' + pet)

In [ ]:
# polars
for pet in pl.Series("values", ['cat', 'dog', 'rabbit']):
    print('I love my ' + pet)

In [ ]:
# datascience
pet = make_array('cat', 'dog', 'rabbit').item(0)
print('I love my ' + pet)

pet = make_array('cat', 'dog', 'rabbit').item(1)
print('I love my ' + pet)

pet = make_array('cat', 'dog', 'rabbit').item(2)
print('I love my ' + pet)

In [ ]:
# polars
pet_pl = pl.Series("values", ['cat', 'dog', 'rabbit']).item(0)
print('I love my ' + pet_pl)

pet_pl = pl.Series("values", ['cat', 'dog', 'rabbit']).item(1)
print('I love my ' + pet_pl)

pet_pl = pl.Series("values", ['cat', 'dog', 'rabbit']).item(2)
print('I love my ' + pet_pl)

In [ ]:
# datascience
np.arange(10)

In [ ]:
# polars
pl.int_range(10, eager=True)

In [ ]:
# datascience
for i in np.arange(10):
    print('GO BEARSSSS')

In [ ]:
# polars
for i in pl.int_range(10, eager=True):
    print('GO BEARSSSS')

(see slides)

In [ ]:
# datascience
game_outcomes = make_array()

for i in np.arange(5):
    game_outcomes = np.append(game_outcomes, simulate_one_round())
    
game_outcomes

In [ ]:
# polars
game_outcomes_pl = []

for i in pl.int_range(5, eager=True):
    game_outcomes_pl.append(simulate_one_round_pl())
    
game_outcomes_pl = pl.Series("values", game_outcomes_pl)
game_outcomes_pl

In [ ]:
# datascience
game_outcomes = make_array()

for i in np.arange(10000):
    game_outcomes = np.append(game_outcomes, simulate_one_round())
    
game_outcomes

In [ ]:
# polars
game_outcomes_pl = []

for i in pl.int_range(10000, eager=True):
    game_outcomes_pl.append(simulate_one_round_pl())
    
game_outcomes_pl = pl.Series("values", game_outcomes_pl)
game_outcomes_pl

In [ ]:
# datascience
len(game_outcomes)

In [ ]:
# polars
len(game_outcomes_pl)

In [ ]:
# datascience
results = Table().with_column('My winnings', game_outcomes)

In [ ]:
# polars
results_pl = pl.DataFrame({'My winnings': game_outcomes_pl})

In [ ]:
# datascience
results

In [ ]:
# polars
results_pl

In [ ]:
# datascience
results.group('My winnings').barh('My winnings')

In [ ]:
# polars
# matplotlib
_df = (lambda _df, _k='My winnings': _df.group_by(_k).len(name='count').sort(_k))(results_pl)
_cat = 'My winnings'
_vals = [c for c in _df.columns if c != _cat and _df[c].dtype.is_numeric()]
if len(_vals) == 1:
    plots.barh(_df.get_column(_cat).to_list(), _df.get_column(_vals[0]).to_list())
    plots.xlabel(_vals[0])
else:
    _n = len(_vals)
    _y = list(range(_df.height))
    _h = 0.8 / _n
    for _i, _c in enumerate(_vals):
        plots.barh([y + _i * _h for y in _y], _df.get_column(_c).to_list(), height=_h, label=_c)
    plots.yticks([y + _h * (_n - 1) / 2 for y in _y], _df.get_column(_cat).to_list())
    plots.legend()

In [ ]:
# datascience
sum(game_outcomes)

In [ ]:
# polars
sum(game_outcomes_pl)

### Another example: simulating heads in 100 coin tosses

In [ ]:
# datascience
coin = make_array('heads', 'tails')

In [ ]:
# polars
coin_pl = pl.Series("values", ['heads', 'tails'])

In [ ]:
# datascience
sum(np.random.choice(coin, 100) == 'heads')

In [ ]:
# polars
sum((coin_pl).sample(100, with_replacement=True) == 'heads')

In [ ]:
# datascience
# Simulate one outcome

def num_heads():
    return sum(np.random.choice(coin, 100) == 'heads')

In [ ]:
# polars
# Simulate one outcome

def num_heads_pl():
    return sum((coin_pl).sample(100, with_replacement=True) == 'heads')

In [ ]:
# Decide how many times you want to repeat the experiment

repetitions = 10000

In [ ]:
# datascience
# Simulate that many outcomes

outcomes = make_array()

for i in np.arange(repetitions):
    outcomes = np.append(outcomes, num_heads())

In [ ]:
# polars
# Simulate that many outcomes

outcomes_pl = []

for i in pl.int_range(repetitions, eager=True):
    outcomes_pl.append(num_heads_pl())
outcomes_pl = pl.Series("values", outcomes_pl)

In [ ]:
# datascience
heads = Table().with_column('Heads', outcomes)
heads.hist(bins = np.arange(29.5, 70.6))

In [ ]:
# polars
# matplotlib
heads_pl = pl.DataFrame({'Heads': outcomes_pl})
_df = heads_pl
_cols = [c for c in _df.columns if _df[c].dtype.is_numeric()]
if len(_cols) == 1:
    plots.hist(_df.get_column(_cols[0]).to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(pl.linear_space(29.5, 70.5, 42, closed="both", eager=True)))
    plots.xlabel(_cols[0])
else:
    for _col in _cols:
        plots.hist(_df.get_column(_col).to_numpy(), density=True, label=_col, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(pl.linear_space(29.5, 70.5, 42, closed="both", eager=True)))
    plots.legend()
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

## `datascience` library

In [ ]:
# datascience
trip.show(3)

In [ ]:
# polars
trip_pl.head(3)

In [ ]:
# datascience
trip.where('Duration', are.above(1000))
# .column('Duration')

In [ ]:
# polars
trip_pl.filter(pl.col('Duration') > 1000)
# .column('Duration')

In [ ]:
# datascience
big_trip_durations = make_array()

In [ ]:
# polars
big_trip_durations_pl = pl.Series("values", [], dtype=pl.Float64)

In [ ]:
# datascience
for duration in trip.column('Duration'):
    if duration > 1000:
        big_trip_durations = np.append(big_trip_durations, duration)
np.mean(big_trip_durations)

In [ ]:
# polars
big_trip_durations_pl = list(big_trip_durations_pl)
for duration in trip_pl.get_column('Duration'):
    if duration > 1000:
        big_trip_durations_pl.append(duration)
big_trip_durations_pl = pl.Series("values", big_trip_durations_pl)
(big_trip_durations_pl).mean()

In [ ]:
# datascience
np.mean(
    trip.where('Duration', are.above(1000)).column('Duration')
)

In [ ]:
# polars
(trip_pl.filter(pl.col('Duration') > 1000).get_column('Duration')).mean()

## Optional: Advanced `where` ##

In [ ]:
# datascience
ages = make_array(16, 22, 18, 15, 19, 15, 16, 21)
age = Table().with_column('Age', ages)

In [ ]:
# polars
ages_pl = pl.Series("values", [16, 22, 18, 15, 19, 15, 16, 21])
age_pl = pl.DataFrame({'Age': ages_pl})

In [ ]:
# datascience
age

In [ ]:
# polars
age_pl

In [ ]:
# datascience
age.where('Age', are.above_or_equal_to(18))

In [ ]:
# polars
age_pl.filter(pl.col('Age') >= 18)

In [ ]:
# datascience
voter = ages >= 18

In [ ]:
# polars
voter_pl = ages_pl >= 18

In [ ]:
# datascience
voter

In [ ]:
# polars
voter_pl

In [ ]:
# datascience
age.where(voter)

In [ ]:
# polars
age_pl.filter(voter_pl)

In [ ]:
# datascience
is_voter = are.above_or_equal_to(18)

In [ ]:
# polars
def is_voter_pl(x):
    return x >= 18

In [ ]:
# datascience
type(is_voter)

In [ ]:
# polars
type(is_voter_pl)

In [ ]:
# datascience
is_voter(22)

In [ ]:
# polars
is_voter_pl(22)

In [ ]:
# datascience
is_voter(3)

In [ ]:
# polars
is_voter_pl(3)

In [ ]:
# datascience
age.apply(is_voter, 'Age')

In [ ]:
# polars
age_pl.get_column('Age').map_elements(is_voter_pl, return_dtype=pl.Boolean)

In [ ]:
# datascience
ages >= 18

In [ ]:
# polars
ages_pl >= 18

In [ ]:
# datascience
voter

In [ ]:
# polars
voter_pl

In [ ]:
def my_voter_function(x):
    return x >= 18

In [ ]:
# datascience
age.where('Age', are.above_or_equal_to(18))

In [ ]:
# polars
age_pl.filter(pl.col('Age') >= 18)

In [ ]:
# datascience
age.where(voter)

In [ ]:
# polars
age_pl.filter(voter_pl)